In [ ]:
import os
import io
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# --- Configuração ---
# 1. Defina o ID da sua "Pasta X" do Google Drive.
#    (Você encontra o ID na URL do Drive: .../folders/ESSE_E_O_ID)
PASTA_DRIVE_ID = "COLOQUE_O_ID_DA_PASTA_X_AQUI" 

# 2. Defina o caminho para sua "Pasta Y" local.
#    (Certifique-se de que esta pasta já exista no seu computador)
PASTA_LOCAL_Y = r"C:\Users\SeuUsuario\Documentos\MinhaPastaLocal" 
# --------------------

# Escopos da API: Define o nível de acesso (leitura/escrita/etc.)
# 'drive.readonly' é suficiente para listar e baixar.
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

def main():
    creds = None
    # O arquivo token.json armazena os tokens de acesso e atualização do usuário.
    # Ele é criado automaticamente na primeira vez que a autenticação é executada.
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # Se não houver credenciais válidas (ou expiradas), solicita o login do usuário.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Carrega o arquivo credentials.json (que você baixou)
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        
        # Salva as credenciais para a próxima execução
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    # Constrói o serviço da API
    service = build('drive', 'v3', credentials=creds)

    print(f"Buscando arquivos na pasta do Drive (ID: {PASTA_DRIVE_ID})...")

    try:
        # 1. LISTAR ARQUIVOS NA PASTA X
        # Monta a query de busca: "'ID_DA_PASTA' in parents"
        query = f"'{PASTA_DRIVE_ID}' in parents"
        results = service.files().list(
            q=query,
            pageSize=100,  # Limite de arquivos por vez
            fields="nextPageToken, files(id, name)"
        ).execute()
        
        items = results.get('files', [])

        if not items:
            print("Nenhum arquivo encontrado na pasta especificada.")
            return

        print(f"Encontrados {len(items)} arquivos. Iniciando transferência para '{PASTA_LOCAL_Y}'...")

        # 2. BAIXAR CADA ARQUIVO PARA A PASTA Y
        for item in items:
            file_id = item['id']
            file_name = item['name']
            caminho_local_completo = os.path.join(PASTA_LOCAL_Y, file_name)

            print(f"  Baixando '{file_name}' (ID: {file_id})...")
            
            # Prepara a requisição de download
            request = service.files().get_media(fileId=file_id)
            
            # Prepara o arquivo local para receber os dados
            fh = io.FileIO(caminho_local_completo, 'wb')
            downloader = MediaIoBaseDownload(fh, request)
            
            done = False
            while done is False:
                status, done = downloader.next_chunk()
                print(f"    Progresso: {int(status.progress() * 100)}%")

            fh.close()
            print(f"  '{file_name}' salvo com sucesso em '{caminho_local_completo}'.")

        print("\nTransferência concluída!")

    except Exception as e:
        print(f"Ocorreu um erro: {e}")

if __name__ == '__main__':
    main()